# LangChain

랭체인(LangChain)은 대규모 언어 모델(LLM)을 활용하여 복잡하고 강력한 애플리케이션을 쉽게 개발할 수 있도록 돕는 오픈소스 프레임워크입니다.  
다양한 LLM(Gemini, OpenAI, HuggingFace 등)을 동일한 코드로 쉽게 교체하며 사용할 수 있는 표준 인터페이스를 제공합니다.  
또한 프롬프트 템플릿을 통해 모델에 내리는 지시사항을 체계적으로 관리합니다.

LangChain Docs: https://docs.langchain.com

## chap08/sec01/langchain_chatbot.ipynb

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-5.6-luna")

response = model.invoke([HumanMessage(content="안녕? 나는 조성민이야.")])
print("AI: " + response.content)

response = model.invoke([HumanMessage(content="내 이름이 뭐지?")])
print("AI: " + response.content)

AI: 안녕하세요, 조성민님! 만나서 반가워요. 무엇을 도와드릴까요?
AI: 아직 이름을 알려주지 않으셔서 모르겠어요. 말씀해 주시면 기억해 둘게요.


## chap08/sec01/langchain_multi_turn.py

In [7]:
from langchain_core.messages import AIMessage, SystemMessage

# 초기 시스템 메시지
messages = [
    # {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},
    SystemMessage("너는 사용자를 도와주는 상담사야."),  
]

while True:
    user_input = input("사용자: ")

    if user_input == "exit":
        break

    # 사용자 메시지를 대화 기록에 추가 
    messages.append(
        # {"role": "user", "content": user_input}
        HumanMessage(user_input)
    )

    print("User: " + user_input)

     # 대화 기록을 기반으로 AI 응답 요청
    ai_response = model.invoke(messages)

    # AI 응답 대화 기록에 추가
    messages.append(
        # {"role": "assistant", "content": ai_response}
        ai_response
    )  

    print("AI: " + ai_response.content)

## 메시지 객체 (Message Object)

랭체인(LangChain)에서는 `SystemMessage`, `HumanMessage`, `AIMessage` 등을 통틀어 **'메시지 객체(Message Object)'**라고 부릅니다. 

이들은 단순한 문자열이나 딕셔너리가 아니라, 랭체인의 최상위 클래스인 `BaseMessage`를 상속받아 생성된 **파이썬 객체(Object)**입니다.

> **핵심 요약**  
> LLM에 데이터를 전달할 때 원시 딕셔너리(Dictionary) 대신 **메시지 객체**를 사용하는 것이 랭체인 설계의 핵심입니다.

<br>

### 메시지 객체가 왜 필요할까? (4가지 이유)

**1. 다양한 LLM 규격 통합 (추상화)**
- OpenAI, Claude, Gemini 등 AI 모델마다 API에서 요구하는 메시지 형식과 역할(Role) 이름이 다릅니다. 
- 메시지 객체를 사용하면 랭체인이 알아서 **각 모델의 규격에 맞게 자동 변환**해주므로, AI 모델을 교체해도 기존 코드를 수정할 필요가 없습니다.

**2. 타입 안정성 확보 및 오류 방지**
- 단순 딕셔너리를 쓸 때 잦은 오타(예: `{"role": "assitant"}`)로 인한 치명적인 런타임 오류를 원천 차단합니다. 
- 파이썬 클래스 기반이므로 **개발 환경(IDE)의 자동 완성 기능**과 **명확한 타입 체크**를 지원받아 코딩이 훨씬 안전해집니다.

**3. 메모리(대화 기록) 시스템과의 완벽한 연동**
- 챗봇의 필수 기능인 과거 대화 내역 저장 및 불러오기 기능(`ConversationBufferMemory` 등)은 단순 딕셔너리가 아닌 **이 메시지 객체들을 기준으로 데이터를 구조화**하도록 설계되어 있습니다.

**4. 복잡한 메타데이터 및 도구(Tool) 처리**
- 최신 LLM이 반환하는 함수 호출(Function Calling) 정보나 토큰 사용량 등의 복잡한 부가 데이터를 다루기 좋습니다.
- 객체 내부에 마련된 `tool_calls`나 `additional_kwargs` 같은 전용 속성을 통해 **데이터를 객체지향적이고 깔끔하게 관리**할 수 있습니다.

<br>

### 원시 API vs 랭체인 방식 비교

| 구분 | 작성 코드 예시 | 장단점 및 특징 |
| :--- | :--- | :--- |
| **원시 API**<br>(OpenAI 등) | `{"role": "assistant", "content": "답변"}` | 직관적이지만 **오타에 취약**하며, 모델 변경 시 **전체 데이터 구조 수정**이 필요함 |
| **랭체인**<br>(LangChain) | `AIMessage(content="답변")` | 어떤 모델을 연결하든 동일한 `BaseMessage` 객체로 통일되어 **유지보수와 확장이 매우 용이함** |

---

## InMemoryChatMessageHistory

InMemoryChatMessageHistory는 대화 과정에서 발생하는 메시지 객체(SystemMessage, HumanMessage, AIMessage 등)들을 저장하는 클래스입니다.  
내부 구조는 사실상 다음과 같습니다.

```python
messages = [
    HumanMessage(content="내 이름은 조성민이야"),
    AIMessage(content="반갑습니다"),
    HumanMessage(content="나에 대해서 알고있는 정보가 있어?"),
    AIMessage(content="반갑습니다"),
    ...
]
```

### 내부 구조
**1. 리스트 기반 적재**
- 내부적으로 파이썬의 리스트(List) 자료구조(messages)를 가지며, 대화의 시계열적 문맥을 순차적으로 기록합니다.

**2. 메시지 객체 캡슐화**
- 사용자의 문자열 입력이나 AI의 텍스트 답변이 들어오면, 이를 랭체인 표준 규격인 BaseMessage 객체로 자동 변환하여 저장합니다.  
이를 통해 다양한 LLM(OpenAI, Claude 등)이 요구하는 이기종 포맷을 단일 규격으로 추상화합니다.

**3. 휘발성 (Volatile)**
- RAM(메모리)에 데이터를 저장하므로 프로그램이 종료되거나 서버가 재시작되면 모든 대화 기록이 소멸합니다.  
때문에 장기 저장이 필요하면 DB 기반 구현이 필요합니다.

## RunnableWithMessageHistory

RunnableWithMessageHistory는 이전 대화 기록을 찾아 프롬프트에 주입하고, 새로운 질문과 LLM의 답변을 다시 저장소에 기록하는 과정을 알아서 처리해 주는 대화 기록 자동 관리 클래스입니다.  
상태(기억)가 없는 기본 LLM 체인을 감싸서 세션별 기억력을 부여해 주기 때문에, 개발자가 매번 과거 데이터를 불러오고 저장하는 번거로운 코드를 작성할 필요 없이 단 한 줄의 코드로 멀티턴 대화를 구현할 수 있게 해줍니다.

**기존 구현**
```python
history = get_session_history("user_1")

response = chain.invoke({
    "chat_history": past_messages, 
    "question": "안녕"
})

history.add_user_message("안녕")
history.add_ai_message(response.content)
]
```

**RunnableWithMessageHistory 사용**
```python
response = chain_with_history.invoke(
    {"question": "안녕"}, 
    config={"configurable": {"session_id": "user_1"}}
)
```

### 생성자(Constructor) 주요 파라미터
이 클래스가 내부 로직을 실행하기 위해 초기화 시점에 다음 인스턴스 변수들을 전달해야 합니다.

- runnable: 내부적으로 invoke()를 호출할 실제 실행 객체입니다. (예: 프롬프트 템플릿 + LLM 체인)
- get_session_history: Session ID(문자열)를 인자로 받아, 해당 세션에 매핑된 상태 저장소 객체(InMemoryChatMessageHistory)를 반환하는 콜백 함수(Callback Function)입니다.
- input_messages_key: 사용자의 입력 문자열을 프롬프트 템플릿의 어떤 파라미터 변수명에 바인딩할지 지정하는 키값입니다.
- history_messages_key: 콜백 함수를 통해 불러온 과거 대화 기록 리스트를 프롬프트 템플릿의 어떤 파라미터 변수명에 바인딩할지 지정하는 키값입니다.

## InMemoryChatMessageHistory와 RunnableWithMessageHistory로 멀티턴 대화 구현 (chap08/sec01/langchain_message_history.ipynb)

In [8]:
from langchain_core.chat_history import InMemoryChatMessageHistory      # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory # 메시지 기록을 활용해 실행 가능한 Wrapper 클래스

# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(
    model, 
    get_session_history,
)

config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성
response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 이성용이야.")],
    config=config,
)

print(response.content)


response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)


config = {"configurable": {"session_id": "abc3"}}
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)


config = {"configurable": {"session_id": "abc2"}}
response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

print(response.content)


config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
    ):
    print(r.content, end="|")

c:\Users\Unreon\Documents\Git\learn_do_it_llm_agent\chap08\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


안녕하세요, 이성용님! 만나서 반가워요. 무엇을 도와드릴까요?
성용님이에요.
아직 이름을 알려주지 않으셨어요.
아까는 성용님이 자기소개를 하셨고, 제가 인사를 드렸어요. 이어서 성용님이 이름을 물어보셔서 제가 “성용님”이라고 답했어요.
|한국|어|를| 사용|하시|고| 이름|이| ‘|이|성|용|’|이|어서| **|대한|민국| 사람|일| 가능|성이| 높|다고| 추|측|**|해| 볼| 수|는| 있지만|,| 이름|과| 언|어|만|으로| 국|적|을| 단|정|할| 수|는| 없|어요|.

|대한|민국| 문화|의| 특징|을| 몇| 가지| 말|하면|:

|-| **|유|교|적| 전|통|**|:| 나|이|와| 직|급|에| 따른| 예|절|,| 어|른| 공|경|,| 교육| 중|시|가| 문화|에| 큰| 영향을| 주|었|어요|.
|-| **|공|동|체| 의|식|**|:| 가족|·|친|구|·|직|장| 공동|체|를| 중요|하게| 여기|며|,| 함께| 식|사|하거나| 모|임|을| 갖|는| 문화|가| 발|달|했|어요|.
|-| **|음|식| 문화|**|:| 김|치|,| 비|빔|밥|,| 불|고|기|,| 국|·|찌|개|처럼| 여러| 사람이| 함께| 나|누|는| 음식|이| 많|고|,| 발|효| 음식|도| 다양|해|요|.
|-| **|현|대| 대|중|문화|**|:| K|-pop|,| 한국| 드|라마|,| 영화|,| 웹|툰|,| 게임| 등이| 세계|적으로| 큰| 영향|력을| 갖|고| 있어|요|.
|-| **|빠|른| 변화|와| 기술| 수|용|**|:| 인터넷|·|모|바일| 기술|이| 일|상|에| 깊|이| 자리| 잡|았|고|,| 전|통|과| 현대| 문화|가| 함께| 존재|해|요|.
|-| **|명|절|과| 전|통|**|:| 설|날|과| 추|석|에| 가족|이| 모|이고|,| 세|배|·|차|례|·|송|편| 같은| 풍|습|을| 경험|하기|도| 해|요|.

|다|만| 한국| 문화|도| 세|대|·|지역|·|개|인|에| 따라| 매우| 다양|하|답|니다|.||||

## Response API 방식과의 차이점

| 비교 항목 | 원시 응답 API (Raw API) | 랭체인 History 아키텍처 |
| :--- | :--- | :--- |
| **상태 관리 주체** | **개발자 제어 (수동)**<br>응답을 받을 때마다 개발자가 직접 배열을 선언하고 `.append()` 메서드를 호출하여 과거 대화 내역을 관리해야 함. | **프레임워크 제어 (자동 캡슐화)**<br>`RunnableWithMessageHistory`의 내부 메서드가 이전 대화 주입 및 새로운 답변 저장을 자동으로 처리함. |
| **세션(사용자) 격리** | **직접 구현 필요**<br>사용자 ID별로 딕셔너리나 DB 테이블을 직접 매핑하고 관리하는 라우팅 로직을 하드코딩해야 함. | **설정 객체 기반 처리**<br>`invoke` 호출 시 `config` 파라미터로 `session_id`만 전달하면 콜백 함수를 통해 독립된 메모리 공간을 격리함. |
| **데이터 모델 포맷** | **제조사 종속적 (JSON/Dict)**<br>OpenAI(`system`, `user`, `assistant`), Claude 등 모델마다 요구하는 JSON Schema 구조가 다름. | **표준화된 객체 지향 모델**<br>`SystemMessage`, `HumanMessage` 등 공통 추상화 객체를 사용하여, 의존성을 분리하고 다형성을 확보함. |
| **데이터베이스 확장성** | **강한 결합도 (Tightly Coupled)**<br>메모리 변수에서 실제 DB(Redis/SQL)로 전환 시, 데이터 삽입/조회 로직이 포함된 메인 실행 코드를 전체적으로 수정해야 함. | **느슨한 결합도 (Loosely Coupled)**<br>`InMemory` 클래스를 `Redis` 등의 클래스로 교체해도 파이프라인(실행 체인) 코드는 수정할 필요가 없는 구조적 유연성을 가짐. |